Imports

In [ ]:
import re
import pickle
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from rake_nltk import Rake

# ----------------------------
# Reproducibility (optional)
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Load CSVs + embeddings

In [ ]:
# ----------------------------
# 1) LOAD DATASETS
# ----------------------------
main_df  = pd.read_csv(r'/teamspace/studios/this_studio/datasets/main_mixed_df.csv')
dialog_df= pd.read_csv(r'/teamspace/studios/this_studio/datasets/dialogue.csv')
docs_df  = pd.read_csv(r'/teamspace/studios/this_studio/datasets/documents.csv')
resp_df  = pd.read_csv(r'/teamspace/studios/this_studio/datasets/response.csv')
summ_df  = pd.read_csv(r'/teamspace/studios/this_studio/datasets/summaries.csv')

# ----------------------------
# 2) LOAD EMBEDDINGS (id -> np.array)
# ----------------------------
def load_embeddings(file_path):
    with open(file_path, 'rb') as f:
        return pickle.load(f)

emb_dialog = load_embeddings(r'/teamspace/studios/this_studio/embeddings/dialog_embeddings.pkl')
emb_docs   = load_embeddings(r'/teamspace/studios/this_studio/embeddings/doc_embeddings.pkl')
emb_resp   = load_embeddings(r'/teamspace/studios/this_studio/embeddings/response_embeddings.pkl')
emb_summ   = load_embeddings(r'/teamspace/studios/this_studio/embeddings/summ_embeddings.pkl')

print("Loaded main_df columns:")
print(list(main_df.columns))
print("Example row:")
print(main_df.iloc[0])

Core helpers: parse sequences, canonical IDs, null handling

In [3]:
# Matches canonical ids like: dl-12, d-7, r-3, s-10
CANON_RE = re.compile(r"^(dl|d|r|s)-\d+$", re.IGNORECASE)

def parse_serialized_sequence(s):
    """
    Parse serialized list strings like:
      '["dl-1","d-2","r-1"]'   OR  '[dl-1,d-2,r-1]'
    into Python list of strings.

    Your data looks like:
      "[dl-1,d-2,r-1,...]"
    so we keep it simple and robust.
    """
    if not isinstance(s, str):
        return []
    s = s.strip()
    if not (s.startswith("[") and s.endswith("]")):
        return []
    items = [x.strip().strip('"').strip("'") for x in s[1:-1].split(",")]
    return [x for x in items if x]

def is_nullish(x):
    """
    True if x is None/NaN/'NULL'/'None'/'' etc.
    """
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    s = str(x).strip().lower()
    return s in {"", "null", "none", "nan"}

def normalize_id(x):
    """
    Standardize id strings (strip spaces).
    """
    return str(x).strip()

def to_canonical(prefix: str, raw_id):
    """
    Prevents double-prefixing:
    - If raw_id already canonical like 'd-1', keep it
    - Else convert to '<prefix>-<raw_id>'
    """
    s = str(raw_id).strip()
    if CANON_RE.match(s):
        return s
    return f"{prefix}-{s}"

Build vocabularies: users, actions, nodes, tasks

In [ ]:
# ----------------------------
# 1) USER VOCAB
# ----------------------------
unique_users = sorted(main_df["userid"].dropna().astype(str).unique())
user2idx = {u: i for i, u in enumerate(unique_users)}
idx2user = {i: u for u, i in user2idx.items()}

# ----------------------------
# 2) ACTION VOCAB (from interactions)
# ----------------------------
all_actions = set()
for seq in main_df["interactions"].dropna():
    all_actions.update(parse_serialized_sequence(seq))

actions2idx = {a: i for i, a in enumerate(sorted(all_actions))}
idx2action  = {i: a for a, i in actions2idx.items()}

# ----------------------------
# 3) NODE VOCAB
#   - from text_nodes sequences
#   - plus query_text_id and target_output_id
# ----------------------------
node_ids = set()

for seq in main_df["text_nodes"].dropna():
    for nid in parse_serialized_sequence(seq):
        if not is_nullish(nid):
            node_ids.add(normalize_id(nid))

for col in ["query_text_id", "target_text_id"]:
    if col in main_df.columns:
        for x in main_df[col].dropna():
            if not is_nullish(x):
                node_ids.add(normalize_id(x))

node2idx = {nid: i for i, nid in enumerate(sorted(node_ids))}
idx2node = {i: nid for nid, i in node2idx.items()}

# ----------------------------
# 4) TASK VOCAB + task_id
# ----------------------------
assert "task" in main_df.columns, "main_df must contain 'task'"
task_names = sorted(main_df["task"].dropna().astype(str).unique())
task2idx = {t: i for i, t in enumerate(task_names)}
idx2task = {i: t for t, i in task2idx.items()}

main_df["task_id"] = main_df["task"].astype(str).map(task2idx)
assert main_df["task_id"].isna().sum() == 0, "Unmapped task found"

n_tasks = len(task2idx)

print("VOCAB SIZES:")
print("  users:", len(user2idx))
print("  actions:", len(actions2idx), "->", sorted(actions2idx.keys()))
print("  nodes:", len(node2idx))
print("  tasks:", n_tasks, "->", task2idx)

Build id2text using canonical IDs (no double-prefix)

In [ ]:
def build_id2text(dialog_df, docs_df, resp_df, summ_df):
    """
    Create mapping canonical_id -> text for ALL text-bearing tables.
    Keys will be like:
      dl-<id>, d-<id>, r-<id>, s-<id>
    """
    id2text = {}

    # Dialogue
    if "dialogue_id" in dialog_df.columns:
        for _, r in dialog_df.dropna(subset=["dialogue_id"]).iterrows():
            cid = to_canonical("dl", r["dialogue_id"])
            id2text[cid] = str(r.get("dialogue_text", ""))

    # Docs
    if "NewsID" in docs_df.columns:
        for _, r in docs_df.dropna(subset=["NewsID"]).iterrows():
            cid = to_canonical("d", r["NewsID"])
            nb = str(r.get("NewsBody", "")).strip()
            hl = str(r.get("Headline", "")).strip()
            id2text[cid] = nb if nb else hl

    # Responses
    if "response_id" in resp_df.columns:
        for _, r in resp_df.dropna(subset=["response_id"]).iterrows():
            cid = to_canonical("r", r["response_id"])
            id2text[cid] = str(r.get("response_text", ""))

    # Summaries
    if "SummID" in summ_df.columns:
        for _, r in summ_df.dropna(subset=["SummID"]).iterrows():
            cid = to_canonical("s", r["SummID"])
            id2text[cid] = str(r.get("Summary", ""))

    return id2text

id2text = build_id2text(dialog_df, docs_df, resp_df, summ_df)

print("id2text size:", len(id2text))
print("sample keys:", list(id2text.keys())[:10])
id2text

In [ ]:
main_df

Offline id2phrases + keyphrase vocabulary kp2idx

In [ ]:
# =========================
# 1. Imports
# =========================

from tqdm.auto import tqdm
import yake
import re

# =========================
# 2. Configuration
# =========================

MAX_KP_PER_TEXT = 50

TEXT_ID_COLS = [
    "query_text_id",
    "target_text_id",
]

YAKE_CONFIG = dict(
    lan="en",
    n=3,                 # unigrams + bigrams
    dedupLim=0.9,
    dedupFunc="seqm",
    windowsSize=1,
    top=MAX_KP_PER_TEXT
)

VALID_PHRASE_REGEX = re.compile(r"^[a-z0-9]+( [a-z0-9]+)*$")

# =========================
# 3. Initialize YAKE
# =========================

kw_extractor = yake.KeywordExtractor(**YAKE_CONFIG)

# =========================
# 4. Collect UNIQUE text ids
# =========================

unique_text_ids = set()

for col in TEXT_ID_COLS:
    if col not in main_df.columns:
        raise ValueError(f"Missing column: {col}")

    unique_text_ids.update(
        main_df[col]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

print(f"Unique text IDs used for KP extraction: {len(unique_text_ids)}")

# =========================
# 5. Build deduplicated id2text
# =========================

id2text_unique = {}

for tid in unique_text_ids:
    if tid not in id2text:
        continue

    text = str(id2text[tid]).strip()
    if text:
        id2text_unique[tid] = text

print(f"Texts actually processed: {len(id2text_unique)}")

# =========================
# 6. YAKE extraction (DEDUPLICATED)
# =========================

id2phrases = {}
kp_vocab = set()

for tid, text in tqdm(
    id2text_unique.items(),
    desc="Extracting YAKE keyphrases (deduplicated)",
    total=len(id2text_unique)
):
    try:
        raw_kps = kw_extractor.extract_keywords(text)
    except Exception:
        id2phrases[tid] = []
        continue

    clean = []
    seen = set()

    for phrase, score in raw_kps:
        p = phrase.lower().strip()

        if p in seen:
            continue

        if not VALID_PHRASE_REGEX.match(p):
            continue

        if len(p) < 3:
            continue

        clean.append(p)
        seen.add(p)

        if len(clean) >= MAX_KP_PER_TEXT:
            break

    id2phrases[tid] = clean
    kp_vocab.update(clean)

# =========================
# 7. Build KP vocabulary
# =========================

kp_vocab = sorted(kp_vocab)

kp2idx = {kp: i for i, kp in enumerate(kp_vocab)}
idx2kp = {i: kp for kp, i in kp2idx.items()}

# =========================
# 8. Sanity checks
# =========================

num_texts = len(id2phrases)
num_empty = sum(1 for v in id2phrases.values() if len(v) == 0)
avg_kps = sum(len(v) for v in id2phrases.values()) / max(num_texts, 1)

print("KP vocab size:", len(kp2idx))
print(f"Texts with 0 KPs: {num_empty} / {num_texts}")
print(f"Average KPs per text: {avg_kps:.2f}")


KP supervision rule: use target_output_id for generation tasks

In [8]:
GEN_TASKS = {"next_item_prediction", "next_user_prediction", "dialogue_generation", "personalized_summarization"}

def get_kp_supervision_id(row):
    """
    For generation tasks:
      - use target_text_id (r-* for dialogue, s-* for summary)
    Else:
      - no KP supervision (return None)
    """
    task = str(row["task"]).strip()
    if task in GEN_TASKS:
        tid = row.get("target_text_id", None)
        if not is_nullish(tid):
            return normalize_id(tid)
    return None

def build_kp_target_from_id(kp_id, id2phrases, kp2idx, device):
    """
    kp_id: canonical id like 'r-2' or 's-10'
    returns (multi_hot_vector, gold_phrase_list)
    """
    phrases = id2phrases.get(kp_id, [])
    y = torch.zeros(len(kp2idx), device=device)
    for p in phrases:
        j = kp2idx.get(p, None)
        if j is not None:
            y[j] = 1.0
    return y, phrases

In [ ]:
from collections import defaultdict

total_rows = len(main_df)

# global counters
rows_with_target = 0
rows_with_valid_kp = 0
rows_zero_kp = 0
rows_missing_text = 0

# task-wise stats
task_stats = defaultdict(lambda: {
    "rows": 0,
    "kp_expected": 0,
    "kp_present": 0,
    "kp_zero": 0,
    "kp_missing": 0,
})

for _, row in main_df.iterrows():
    task = str(row["task"]).strip()
    task_stats[task]["rows"] += 1

    tid = row.get("target_text_id", None)
    if tid is None or str(tid).strip() == "":
        continue  # no KP expected

    tid = normalize_id(tid)
    rows_with_target += 1
    task_stats[task]["kp_expected"] += 1

    if tid not in id2phrases:
        rows_missing_text += 1
        task_stats[task]["kp_missing"] += 1
        continue

    if len(id2phrases[tid]) == 0:
        rows_zero_kp += 1
        task_stats[task]["kp_zero"] += 1
        continue

    rows_with_valid_kp += 1
    task_stats[task]["kp_present"] += 1


# ----------------------------
# GLOBAL SUMMARY
# ----------------------------
print("\n========== GLOBAL KP COVERAGE ==========")
print(f"Total rows                : {total_rows}")
print(f"Rows with target_text_id  : {rows_with_target}")
print(f"Rows with VALID KP        : {rows_with_valid_kp}")
print(f"Rows with ZERO KP         : {rows_zero_kp}")
print(f"Rows with MISSING text    : {rows_missing_text}")

if rows_with_target > 0:
    print(
        f"Effective KP supervision : "
        f"{rows_with_valid_kp / rows_with_target:.3f}"
    )

# ----------------------------
# TASK-WISE SUMMARY
# ----------------------------
print("\n========== TASK-WISE KP COVERAGE ==========")

for task, s in sorted(task_stats.items()):
    if s["kp_expected"] == 0:
        continue

    eff = s["kp_present"] / max(s["kp_expected"], 1)

    print(
        f"{task:30s} | "
        f"rows={s['rows']:6d} | "
        f"kp_expected={s['kp_expected']:6d} | "
        f"kp_valid={s['kp_present']:6d} | "
        f"kp_zero={s['kp_zero']:6d} | "
        f"kp_missing={s['kp_missing']:6d} | "
        f"eff={eff:.3f}"
    )


In [ ]:
def has_valid_kp(row):
    tid = row.get("target_text_id", None)
    if tid is None or str(tid).strip() == "":
        return False

    tid = normalize_id(tid)

    if tid not in id2phrases:
        return False

    if len(id2phrases[tid]) == 0:
        return False

    return True


before = len(main_df)

main_df = main_df[main_df.apply(has_valid_kp, axis=1)].reset_index(drop=True)

after = len(main_df)

print("Dropped rows:", before - after)
print("Remaining rows:", after)


Embedding resolver for canonical ids

In [11]:
class EmbLookup:
    @staticmethod
    def lookup(emb_dict, key):
        if key in emb_dict:
            return emb_dict[key]
        s = str(key)
        if s in emb_dict:
            return emb_dict[s]
        # numeric tail fallback
        nums = re.findall(r"\d+", s)
        if nums:
            n = nums[-1]
            if n in emb_dict:
                return emb_dict[n]
            try:
                ni = int(n)
                if ni in emb_dict:
                    return emb_dict[ni]
            except Exception:
                pass
        return None

class CanonicalEmbeddingResolver:
    """
    Input: canonical id string (dl-7, d-3, r-2, s-10)
    Output: np.array shape (768,)
    """
    def __init__(self, emb_dialog, emb_docs, emb_resp, emb_summ, d=768):
        self.emb_dialog = emb_dialog
        self.emb_docs = emb_docs
        self.emb_resp = emb_resp
        self.emb_summ = emb_summ
        self.d = d

    def get(self, cid: str) -> np.ndarray:
        cid = normalize_id(cid)
        prefix = cid.split("-")[0].lower()

        if prefix == "dl":
            v = EmbLookup.lookup(self.emb_dialog, cid)
        elif prefix == "d":
            v = EmbLookup.lookup(self.emb_docs, cid)
        elif prefix == "r":
            v = EmbLookup.lookup(self.emb_resp, cid)
        elif prefix == "s":
            v = EmbLookup.lookup(self.emb_summ, cid)
        else:
            v = None

        if v is None:
            return np.zeros((self.d,), dtype=np.float32)

        v = np.asarray(v, dtype=np.float32)
        if v.shape[-1] != self.d:
            raise ValueError(f"Embedding dim mismatch for {cid}: got {v.shape}, expected ({self.d},)")
        return v

emb_resolver = CanonicalEmbeddingResolver(emb_dialog, emb_docs, emb_resp, emb_summ, d=768)

UniHis SSM Encoder

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class UniHisSSMEncoder(nn.Module):
    def __init__(
        self,
        vocab_sizes,
        d_model=768,
        d_state=64,
        n_tasks=8,
        n_users=10000,
        d_qtext=768,
        known_task=True
    ):
        super().__init__()

        self.d_state = d_state
        self.known_task = known_task

        # ========= SSM EIGEN PARAMS =========
        def init_eig():
            Q = nn.Parameter(torch.randn(d_state, d_state) * 0.02)
            Lambda = nn.Parameter(torch.empty(d_state).uniform_(-1.0, -0.1))
            return Q, Lambda

        self.Q_L, self.Lambda_L = init_eig()
        self.Q_S, self.Lambda_S = init_eig()
        self.Q_E, self.Lambda_E = init_eig()

        self.B = nn.Parameter(torch.randn(d_state, d_model))

        self.tau_S = nn.Parameter(torch.tensor(1.0))
        self.tau_E = nn.Parameter(torch.tensor(1.0))

        # ========= ACTION → MOOD =========
        self.action_proj = nn.Linear(vocab_sizes["actions"], d_model)

        self.MLP_mood = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )

        self.W_a = nn.Linear(d_model, d_state)

        # ========= MIXERS =========
        self.Mixer_L = nn.Linear(d_state, d_state)
        self.Mixer_S = nn.Linear(d_state, d_state)
        self.Mixer_E = nn.Linear(d_state, d_state)

        self.W_fuse = nn.Linear(3 * d_state, 3)
        self.Mixer_Fuse = nn.Linear(d_state, d_state)

        # ========= HEADS =========
        self.action_head = nn.Linear(3 * d_state, vocab_sizes["actions"])
        self.node_head   = nn.Linear(3 * d_state, vocab_sizes["nodes"])

        # ========= TASK =========
        self.Embed_Q = nn.Linear(d_qtext, d_state)
        self.W_task  = nn.Linear(2 * d_state, n_tasks)
        self.log_var_task = nn.Parameter(torch.zeros(()))

        self.W_hyp = nn.Linear(2 * d_state, d_state)
        self.W_theta = nn.ModuleList([
            nn.Linear(d_state, 2 * d_state) for _ in range(n_tasks)
        ])

        # ========= USER =========
        self.W_user = nn.Linear(d_state, n_users)

        self.criterion = nn.CrossEntropyLoss()

    # ========= SSM MATRIX =========
    def make_A(self, Q, Lambda_diag):
        Q_inv = torch.linalg.inv(Q)
        return Q @ torch.diag(Lambda_diag) @ Q_inv

    # ========= FORWARD =========
    def forward(
        self,
        e_seq,             # (B, T, d_model)
        action_onehot,     # (B, T, |A|)
        action_labels,     # (B, T)
        node_labels,       # (B, T)
        task_labels,       # (B,)
        user_labels,       # (B,)
        q_text_emb         # (B, d_qtext)
    ):
        B, T, _ = e_seq.shape
        device = e_seq.device

        # ========= INITIAL STATES =========
        xL = torch.zeros(B, self.d_state, device=device)
        xS = torch.zeros(B, self.d_state, device=device)
        xE = torch.zeros(B, self.d_state, device=device)

        act_logits = []
        node_logits = []

        # ========= SSM MATRICES =========
        A_L = self.make_A(self.Q_L, torch.exp(self.Lambda_L))

        eps_S = torch.sigmoid(self.tau_S)
        k = torch.log((1 - eps_S) / eps_S) / torch.abs(self.Lambda_S)
        A_S = self.make_A(self.Q_S, torch.sigmoid(-k * self.Lambda_S))

        eta = (torch.exp(-1.0 / self.tau_E) - 1.0) / self.Lambda_E
        A_E = self.make_A(self.Q_E, 1.0 + eta * self.Lambda_E)

        # ========= SEQUENCE LOOP =========
        for t in range(T):
            e_t = e_seq[:, t]                      # (B, d_model)
            a_t = action_onehot[:, t, :]           # (B, |A|)

            a_emb = self.action_proj(a_t)
            mood_vec = self.MLP_mood(a_emb)
            mood = torch.sigmoid(self.W_a(mood_vec))

            u_t = torch.bmm(
                (self.B.unsqueeze(0) * mood.unsqueeze(2)),
                e_t.unsqueeze(2)
            ).squeeze(2)

            xL = torch.matmul(xL, A_L.T) + u_t
            xS = torch.matmul(xS, A_S.T) + u_t
            xE = torch.matmul(xE, A_E.T) + u_t

            x_cat = torch.cat([xL, xS, xE], dim=-1)

            act_logits.append(self.action_head(x_cat))
            node_logits.append(self.node_head(x_cat))

        act_logits  = torch.stack(act_logits, dim=1)     # (B, T, |A|)
        node_logits = torch.stack(node_logits, dim=1)    # (B, T, |V|)

        last_node_logits = node_logits[:, -1]
        last_node_pred   = last_node_logits.argmax(dim=-1)

        # ========= HISTORY FUSION =========
        yL = self.Mixer_L(xL)
        yS = self.Mixer_S(xS)
        yE = self.Mixer_E(xE)

        w = F.softmax(self.W_fuse(torch.cat([xL, xS, xE], dim=-1)), dim=-1)
        h_hist = self.Mixer_Fuse(
            w[:, 0:1] * yL +
            w[:, 1:2] * yS +
            w[:, 2:3] * yE
        )

        # ========= ACTION + NODE LOSS =========
        loss_action = self.criterion(
            act_logits.view(-1, act_logits.size(-1)),
            action_labels.view(-1)
        )

        loss_node = self.criterion(
            node_logits.view(-1, node_logits.size(-1)),
            node_labels.view(-1)
        )

        # ========= TASK CONDITIONING =========
        q_emb = self.Embed_Q(q_text_emb)
        z_switch = torch.cat([h_hist, q_emb], dim=-1)

        task_logits = self.W_task(z_switch)
        P_task = F.softmax(task_logits, dim=-1)

        ce_task = self.criterion(task_logits, task_labels)
        loss_task = ce_task

        h_hyp = F.relu(self.W_hyp(z_switch))

        if self.known_task:
            theta = self.W_theta[task_labels[0]](h_hyp)
            g, b = theta.chunk(2, dim=-1)
            h_final = torch.sigmoid(g) * h_hist + b
        else:
            h_final = 0.0
            for i in range(len(self.W_theta)):
                theta_i = self.W_theta[i](h_hyp)
                g_i, b_i = theta_i.chunk(2, dim=-1)
                h_final = h_final + P_task[:, i:i+1] * (
                    torch.sigmoid(g_i) * h_hist + b_i
                )

        # ========= USER LOSS =========
        user_logits = self.W_user(h_final)
        P_user = F.softmax(user_logits, dim=-1)
        loss_user = self.criterion(user_logits, user_labels)

        # ========= TOTAL LOSS =========
        total_loss = loss_action + loss_node + loss_task + loss_user

        return {
            "loss": total_loss,

            "loss_action": loss_action,
            "loss_node": loss_node,
            "loss_task": loss_task,
            "loss_user": loss_user,

            "node_logits": node_logits,
            "pred_nodes": node_logits.argmax(dim=-1),
            "last_node_logits": last_node_logits,
            "last_node_pred": last_node_pred,

            "task_logits": task_logits,
            "P_task": P_task,
            "pred_task": P_task.argmax(dim=-1),
            "true_task": task_labels,

            "user_logits": user_logits,
            "P_user": P_user,
            "pred_user": P_user.argmax(dim=-1),
            "true_user": user_labels,

            "h_final": h_final
        }


RWKV keyphrase decoder + combined model

In [13]:
class LensGate(nn.Module):
    def __init__(self, d_state, d_qtext):
        super().__init__()
        self.Wq = nn.Linear(d_qtext, d_state)
        self.Wfilm = nn.Linear(d_state, 2 * d_state)
        self.Wg = nn.Linear(2 * d_state, d_state)

    def forward(self, h_final, q_text_emb):
        q0 = self.Wq(q_text_emb)
        gamma, beta = self.Wfilm(h_final).chunk(2, dim=-1)
        q_lens = (1.0 + gamma) * q0 + beta
        g = torch.sigmoid(self.Wg(torch.cat([h_final, q_lens], dim=-1)))
        return g * q_lens + (1.0 - g) * q0


class RWKVTimeMix(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.time_mix = nn.Parameter(torch.rand(d))
        self.key = nn.Linear(d, d, bias=False)
        self.value = nn.Linear(d, d, bias=False)
        self.receptance = nn.Linear(d, d, bias=False)
        self.output = nn.Linear(d, d, bias=False)
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        x = self.ln(x)
        x_shift = torch.zeros_like(x)
        x_shift[:, 1:] = x[:, :-1]
        xm = x * self.time_mix + x_shift * (1 - self.time_mix)

        r = torch.sigmoid(self.receptance(xm))
        k = self.key(xm)
        v = self.value(xm)

        w = torch.softmax(k, dim=1)
        mix = torch.sum(w * v, dim=1, keepdim=True)
        y = self.output(r * mix).expand_as(x)
        return x + y


class RWKVChannelMix(nn.Module):
    def __init__(self, d, expansion=4):
        super().__init__()
        self.time_mix = nn.Parameter(torch.rand(d))
        self.key = nn.Linear(d, expansion * d, bias=False)
        self.value = nn.Linear(expansion * d, d, bias=False)
        self.receptance = nn.Linear(d, d, bias=False)
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        x = self.ln(x)
        x_shift = torch.zeros_like(x)
        x_shift[:, 1:] = x[:, :-1]
        xm = x * self.time_mix + x_shift * (1 - self.time_mix)

        r = torch.sigmoid(self.receptance(xm))
        k = torch.relu(self.key(xm))
        v = self.value(k)
        return x + r * v
class RWKVKeyphraseDecoder(nn.Module):
    def __init__(self, d_state, d_qtext, n_tasks, n_keyphrases, n_layers=2):
        super().__init__()

        self.lens_gate = LensGate(d_state, d_qtext)

        self.task_embed = nn.Embedding(n_tasks, d_state)
        self.task_proj = nn.Linear(d_state, d_state)

        self.proj_h = nn.Linear(d_state, d_state)
        self.proj_q = nn.Linear(d_state, d_state)

        self.blocks = nn.ModuleList()
        for _ in range(n_layers):
            self.blocks.append(RWKVTimeMix(d_state))
            self.blocks.append(RWKVChannelMix(d_state))

        self.out_ln = nn.LayerNorm(d_state)
        self.kp_head = nn.Linear(d_state, n_keyphrases)

        self.bce = nn.BCEWithLogitsLoss(reduction="none")

    def forward(self, h_final, q_text_emb, P_task, kp_targets, kp_loss_mask):
        B = h_final.size(0)

        q_cond = self.lens_gate(h_final, q_text_emb)

        task_ids = torch.arange(P_task.size(1), device=h_final.device)
        Temb = self.task_embed(task_ids)
        prompt = self.task_proj(P_task @ Temb)

        token_h = self.proj_h(h_final + prompt)
        token_q = self.proj_q(q_cond + prompt)

        x = torch.stack([token_h, token_q], dim=1)  # (B,2,d)

        for blk in self.blocks:
            x = blk(x)

        h_dec = self.out_ln(x[:, -1])
        kp_logits = self.kp_head(h_dec)

        loss_kp = self.bce(kp_logits, kp_targets).mean(dim=-1)
        loss_kp = (loss_kp * kp_loss_mask).mean()

        return {
            "kp_logits": kp_logits,
            "loss_kp": loss_kp
        }

class UniHisSSMWithKeyphrases(nn.Module):
    def __init__(self, encoder, d_qtext, n_tasks, n_keyphrases, rwkv_layers=2):
        super().__init__()

        self.encoder = encoder
        self.decoder = RWKVKeyphraseDecoder(
            d_state=encoder.d_state,
            d_qtext=d_qtext,
            n_tasks=n_tasks,
            n_keyphrases=n_keyphrases,
            n_layers=rwkv_layers
        )

    def forward(
        self,
        e_seq,
        action_onehot,
        action_labels,
        node_labels,
        task_labels,
        user_labels,
        q_text_emb,
        kp_targets,
        kp_loss_mask
    ):
        enc_out = self.encoder(
            e_seq=e_seq,
            action_onehot=action_onehot,
            action_labels=action_labels,
            node_labels=node_labels,
            task_labels=task_labels,
            user_labels=user_labels,
            q_text_emb=q_text_emb
        )

        dec_out = self.decoder(
            h_final=enc_out["h_final"],
            q_text_emb=q_text_emb,
            P_task=enc_out["P_task"],
            kp_targets=kp_targets,
            kp_loss_mask=kp_loss_mask
        )

        total_loss = (
            enc_out["loss"]
            + dec_out["loss_kp"]
        )

        return {
            **enc_out,
            **dec_out,
            "loss_total": total_loss
        }


Instantiate model correctly

In [ ]:
# Encoder head sizes
vocab_sizes = {
    "actions": len(actions2idx),
    "nodes": len(node2idx),
}

n_users_model = len(user2idx)  # IMPORTANT: avoid out-of-range user IDs

encoder = UniHisSSMEncoder(
    vocab_sizes=vocab_sizes,
    d_model=768,
    d_state=64,
    n_tasks=n_tasks,
    n_users=n_users_model,
    d_qtext=768,
    known_task=False
)

model = UniHisSSMWithKeyphrases(
    encoder=encoder,
    d_qtext=768,
    n_tasks=n_tasks,
    n_keyphrases=len(kp2idx),
    rwkv_layers=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=3e-4)

print("Model ready on:", device)


Sample 1-row forward pass (uses text_nodes + correct KP supervision)

In [ ]:
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------
MAX_LEN = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------------------------------
# SAFE PARSERS
# -------------------------------------------------------
def parse_serialized_sequence(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return x
    try:
        v = ast.literal_eval(x)
        return v if isinstance(v, list) else []
    except Exception:
        return []

def normalize_id(x):
    return str(x).strip()

# -------------------------------------------------------
# FIX SEQUENCE LENGTHS (<=100 and MATCHED)
# -------------------------------------------------------
def fix_and_align_sequences(nodes, actions, max_len=100):
    nodes   = list(nodes)
    actions = list(actions)
    L = min(len(nodes), len(actions), max_len)
    return nodes[:L], actions[:L]

@torch.no_grad()
def decode_kp_logits(
    kp_logits,
    idx2kp,
    topk=15,
    threshold=0.3,
    apply_sigmoid=True
):
    if apply_sigmoid:
        scores = torch.sigmoid(kp_logits)
    else:
        scores = kp_logits

    all_preds = []
    for b in range(scores.size(0)):
        s = scores[b]
        keep = s >= threshold
        idxs = torch.nonzero(keep).squeeze(-1)

        if idxs.numel() == 0:
            all_preds.append([])
            continue

        vals = s[idxs]
        k = min(topk, vals.numel())
        topv, topi = torch.topk(vals, k)

        phrases = [
            (idx2kp[int(idxs[i])], float(topv[i].item()))
            for i in range(k)
        ]
        all_preds.append(phrases)

    return all_preds

# -------------------------------------------------------
# ENSURE QUERY / TARGET IDS
# -------------------------------------------------------
def resolve_target_ids(row):
    task = str(row["task"]).strip().lower()
    qid = normalize_id(row["query_text_id"])

    if task in ["next_user_prediction", "next_item_prediction"]:
        return qid, qid
    else:
        return normalize_id(row["query_text_id"]), normalize_id(row["target_text_id"])

# -------------------------------------------------------
# RUN SANITY CHECK PIPELINE FOR 1 SAMPLE PER TASK
# -------------------------------------------------------
sampled_rows = (
    main_df
    .groupby("task", sort=False)
    .head(1)
    .reset_index(drop=True)
)

for _, row in sampled_rows.iterrows():

    userid  = str(row["userid"])
    task    = str(row["task"]).strip()
    task_id = int(row["task_id"])

    # -------------------------------------------------------
    # PARSE + FIX SEQUENCES
    # -------------------------------------------------------
    raw_nodes   = parse_serialized_sequence(row["text_nodes"])
    raw_actions = parse_serialized_sequence(row["interactions"])

    raw_nodes, raw_actions = fix_and_align_sequences(
        raw_nodes, raw_actions, MAX_LEN
    )

    if len(raw_nodes) == 0:
        raise ValueError("Empty sequence after fixing.")

    T = len(raw_nodes)

    # -------------------------------------------------------
    # ACTION IDS
    # -------------------------------------------------------
    action_ids = [actions2idx[a] for a in raw_actions]
    action_labels = torch.tensor(
        action_ids, device=device
    ).unsqueeze(0)

    # -------------------------------------------------------
    # NODE IDS
    # -------------------------------------------------------
    node_seq = [normalize_id(n) for n in raw_nodes]
    node_label_ids = [node2idx[n] for n in node_seq]
    node_labels = torch.tensor(
        node_label_ids, device=device
    ).unsqueeze(0)

    # -------------------------------------------------------
    # NODE EMBEDDINGS
    # -------------------------------------------------------
    e_seq_np = np.stack(
        [emb_resolver.get(n) for n in node_seq], axis=0
    )
    e_seq = torch.tensor(
        e_seq_np, device=device
    ).unsqueeze(0)

    # -------------------------------------------------------
    # QUERY / TARGET IDS
    # -------------------------------------------------------
    target_text_id, next_item_text_id = resolve_target_ids(row)

    query_text_id = normalize_id(row["query_text_id"])
    true_last_node_id = node2idx[query_text_id]
    true_last_node = torch.tensor(
        [true_last_node_id], device=device
    )

    q_emb_np = emb_resolver.get(query_text_id)
    q_text_emb = torch.tensor(
        q_emb_np, device=device
    ).unsqueeze(0)

    # -------------------------------------------------------
    # USER / TASK LABELS
    # -------------------------------------------------------
    user_labels = torch.tensor(
        [user2idx[userid]], device=device
    )
    task_labels = torch.tensor(
        [task_id], device=device
    )

    # -------------------------------------------------------
    # KP SUPERVISION
    # -------------------------------------------------------
    kp_id = get_kp_supervision_id(row)

    if kp_id is None:
        kp_targets = torch.zeros(
            1, len(kp2idx), device=device
        )
        kp_mask = torch.zeros(1, device=device)
        gold_phrases = []
    else:
        kp_y, gold_phrases = build_kp_target_from_id(
            kp_id, id2phrases, kp2idx, device=device
        )
        kp_targets = kp_y.unsqueeze(0)
        kp_mask = torch.ones(1, device=device)

    # -------------------------------------------------------
    # MODEL FORWARD
    # -------------------------------------------------------
    model.train()

    action_onehot = F.one_hot(
        action_labels, num_classes=len(actions2idx)
    ).float()

    out = model(
        e_seq=e_seq,
        action_onehot=action_onehot,
        action_labels=action_labels,
        node_labels=node_labels,
        task_labels=task_labels,
        user_labels=user_labels,
        q_text_emb=q_text_emb,
        kp_targets=kp_targets,
        kp_loss_mask=kp_mask
    )

    optim.zero_grad()
    out["loss_total"].backward()
    optim.step()

    # -------------------------------------------------------
    # DECODE OUTPUTS
    # -------------------------------------------------------
    pred_last_node_id = int(out["last_node_pred"].item())
    true_last_node_id = int(true_last_node.item())

    pred_last_node_tok = idx2node[pred_last_node_id]
    true_last_node_tok = idx2node[true_last_node_id]
    last_node_correct = pred_last_node_id == true_last_node_id

    # -------------------------------------------------------
    # NEXT ITEM TOP-15
    # -------------------------------------------------------
    item_logits = out["last_node_logits"].detach()
    item_probs = torch.softmax(item_logits, dim=-1)

    top_item_probs, top_item_ids = torch.topk(item_probs, k=15, dim=-1)
    top_item_ids = top_item_ids[0].tolist()
    top_item_probs = top_item_probs[0].tolist()

    true_item_rank = None
    for i, nid in enumerate(top_item_ids):
        if nid == true_last_node_id:
            true_item_rank = i + 1
            break

    # -------------------------------------------------------
    # DECODE KP
    # -------------------------------------------------------
    model.eval()
    pred_phrases = decode_kp_logits(
        out["kp_logits"],
        idx2kp=idx2kp,
        topk=15,
        threshold=0.03
    )

    # -------------------------------------------------------
    # PRINT SUMMARY
    # -------------------------------------------------------
    print("\n========== SAMPLE ==========")
    print("User:", userid)
    print("Task:", task)

    true_user_id = int(out["true_user"].item())
    pred_user_id = int(out["pred_user"].item())

    user_logits = out["user_logits"].detach()
    user_probs = torch.softmax(user_logits, dim=-1)

    top_user_probs, top_user_ids = torch.topk(user_probs, k=15, dim=-1)
    top_user_ids = top_user_ids[0].tolist()
    top_user_probs = top_user_probs[0].tolist()

    true_user_rank = None
    for i, uid in enumerate(top_user_ids):
        if uid == true_user_id:
            true_user_rank = i + 1
            break

    print("\nUSER PREDICTION:")
    print("  TRUE USER:", idx2user[true_user_id])
    print("  PRED USER:", idx2user[pred_user_id])
    print("  MATCH    :", true_user_id == pred_user_id)

    print("\nSequence length:", T)

    print("\nNEXT ITEM PREDICTION:")
    print("  TRUE ITEM:", true_last_node_tok)
    print("  PRED ITEM:", pred_last_node_tok)
    print("  MATCH    :", last_node_correct)

    print("\nKP GOLD (top 15):")
    print(gold_phrases[:15])

    print("\nKP PRED:")
    for ph, pr in pred_phrases[0]:
        print(f"  {ph:30s} {pr:.4f}")

    print("\nLOSSES:")
    for k in ["loss_action", "loss_node", "loss_user", "loss_task", "loss_kp", "loss_total"]:
        if k in out:
            print(f"{k:15s}:", float(out[k].item()))

    print("\n" + "=" * 80 + "\n")


rain/test split (70–30) + basic setup

In [ ]:
from tqdm.auto import tqdm
import os
import math

# ----------------------------
# 70-30 split
# ----------------------------
rng = np.random.default_rng(42)
idx = np.arange(len(main_df))
rng.shuffle(idx)

# Ensure split column is normalized
main_df["split"] = main_df["split"].str.strip().str.lower()

# Create train and test DataFrames
train_df = main_df[main_df["split"] == "train"].reset_index(drop=True)
test_df  = main_df[main_df["split"] == "test"].reset_index(drop=True)

print("Train size:", len(train_df), "| Test size:", len(test_df))


# print("Train size:", len(train_df), "| Test size:", len(test_df))

# ----------------------------
# Checkpoint folder
# ----------------------------
CKPT_DIR = r"/teamspace/studios/this_studio/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoint dir:", CKPT_DIR)

Batch builder (row → tensors)

In [22]:
# ===============================
# DATA LOADER (NO NONE ANYWHERE)
# ===============================

MAX_T = 100

def build_row_tensors(row, device):
    """
    HARD GUARANTEES:
    - kp_targets is ALWAYS a tensor
    - kp_loss_mask decides whether KP loss contributes
    - user/node losses always active
    """

    # ---------- user ----------
    userid = str(row["userid"]).strip()
    if userid not in user2idx:
        raise ValueError(f"unknown userid: {userid}")

    user_labels = torch.tensor([user2idx[userid]], device=device)  # (1,)

    # ---------- task ----------
    task_id = int(row["task_id"])
    task_labels = torch.tensor([task_id], device=device)  # (1,)

    # ---------- parse sequences ----------
    actions = parse_serialized_sequence(row["interactions"])
    nodes   = parse_serialized_sequence(row["text_nodes"])

    if len(actions) == 0 or len(nodes) == 0:
        raise ValueError("empty sequence")
    if len(actions) != len(nodes):
        raise ValueError("length mismatch")

    # keep last MAX_T
    if len(actions) > MAX_T:
        actions = actions[-MAX_T:]
        nodes   = nodes[-MAX_T:]

    T = len(actions)

    # ---------- action labels ----------
    action_ids = [actions2idx[a] for a in actions]
    action_labels = torch.tensor(action_ids, device=device).unsqueeze(0)  # (1,T)

    action_onehot = F.one_hot(
        action_labels, num_classes=len(actions2idx)
    ).float()  # (1,T,A)

    # ---------- node labels ----------
    node_ids = [node2idx[normalize_id(n)] for n in nodes]
    node_labels = torch.tensor(node_ids, device=device).unsqueeze(0)  # (1,T)

    # ---------- node embeddings ----------
    e_seq = torch.tensor(
        np.stack([emb_resolver.get(normalize_id(n)) for n in nodes]),
        device=device
    ).unsqueeze(0)  # (1,T,768)

    # ---------- query embedding ----------
    qid = normalize_id(row["query_text_id"])
    q_text_emb = torch.tensor(
        emb_resolver.get(qid), device=device
    ).unsqueeze(0)  # (1,768)

    # ---------- KP supervision (NO NONE) ----------
    kp_targets = torch.zeros(
        1, len(kp2idx), device=device
    )  # ALWAYS exists

    kp_mask = torch.tensor([0.0], device=device)

    kp_id = get_kp_supervision_id(row)
    if kp_id is not None and kp_id in id2phrases:
        y, _ = build_kp_target_from_id(
            kp_id, id2phrases, kp2idx, device=device
        )
        kp_targets[0] = y
        kp_mask[0] = 1.0

    return {
        "e_seq": e_seq,
        "action_onehot": action_onehot,
        "action_labels": action_labels,
        "node_labels": node_labels,
        "user_labels": user_labels,
        "task_labels": task_labels,
        "q_text_emb": q_text_emb,
        "kp_targets": kp_targets,
        "kp_loss_mask": kp_mask,
    }


In [ ]:
import os

CKPT_DIR = "/teamspace/studios/this_studio/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

SAVE_EVERY_STEPS = 2000   # step-based safety
SAVE_EVERY_EPOCH = True

RESUME_FROM = None
# e.g. RESUME_FROM = "checkpoints/epoch_2.pt"




start_epoch = 1

if RESUME_FROM is not None:
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt["model"])
    optim.load_state_dict(ckpt["optim"])
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from {RESUME_FROM}, starting at epoch {start_epoch}")

# ===============================
# TRAINING LOOP (BALANCED + CKPT)
# ===============================

EPOCHS = 4
CLIP_NORM = 0.75

LOSS_KEYS = [
    "loss_total",
    "loss_user",
    "loss_node",
    "loss_kp",
]

LOSS_WEIGHTS = {
    "loss_user": 0.1,
    "loss_node": 0.1,
    "loss_kp":   5.0,
}

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()

    running = {k: 0.0 for k in LOSS_KEYS}
    steps_ok = 0
    steps_skip = 0
    all_targets_available = True

    pbar = tqdm(range(len(train_df)), desc=f"Epoch {epoch}")

    for i in pbar:
        row = train_df.iloc[i]

        try:
            batch = build_row_tensors(row, device)
        except Exception:
            steps_skip += 1
            all_targets_available = False
            continue

        if batch["kp_loss_mask"].item() != 1.0:
            all_targets_available = False

        out = model(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            user_labels=batch["user_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"],
            kp_targets=batch["kp_targets"],
            kp_loss_mask=batch["kp_loss_mask"],
        )

        # ---------- BALANCED LOSS ----------
        loss = 0.0
        for k, v in out.items():
            if k.startswith("loss_"):
                w = LOSS_WEIGHTS.get(k, 1.0)
                loss = loss + w * v

        if not torch.isfinite(loss):
            optim.zero_grad(set_to_none=True)
            steps_skip += 1
            all_targets_available = False
            continue

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        optim.step()

        steps_ok += 1

        running["loss_total"] += float(loss.item())
        for k in LOSS_KEYS:
            if k != "loss_total":
                running[k] += float(out[k].item())

        # ---------- STEP CHECKPOINT ----------
        if steps_ok % SAVE_EVERY_STEPS == 0:
            torch.save({
                "epoch": epoch,
                "step": steps_ok,
                "model": model.state_dict(),
                "optim": optim.state_dict(),
            }, os.path.join(CKPT_DIR, f"step_{epoch}_{steps_ok}.pt"))

        denom = max(1, steps_ok)
        pbar.set_postfix({
            "L": f"{running['loss_total']/denom:.3f}",
            "Lu": f"{running['loss_user']/denom:.3f}",
            "Ln": f"{running['loss_node']/denom:.3f}",
            "Lkp": f"{running['loss_kp']/denom:.3f}",
            "skip": steps_skip
        })

    # ---------- EPOCH CHECKPOINT ----------
    if SAVE_EVERY_EPOCH:
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optim": optim.state_dict(),
            "epoch_losses": {
                k: running[k] / max(1, steps_ok)
                for k in LOSS_KEYS
            },
        }, os.path.join(CKPT_DIR, f"epoch_{epoch}.pt"))

    print(f"\nEpoch {epoch} SUMMARY")
    for k in LOSS_KEYS:
        print(f"{k:12s}: {running[k]/max(1,steps_ok):.4f}")
    print(f"steps_ok={steps_ok}  skipped={steps_skip}")

    if all_targets_available:
        print("👍 All targets available for every row in this epoch")
    else:
        print("⚠️ Some rows missing targets")


In [ ]:
import torch
import pandas as pd
from tqdm import tqdm

CKPT_PATH = "path/to/latest_checkpoint.pt"   # <-- set this

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()

records = []

TOPK = 10

with torch.no_grad():
    for i in tqdm(range(len(test_df)), desc="Inference"):
        row = test_df.iloc[i]

        try:
            batch = build_row_tensors(row, device)
        except Exception:
            continue

        out = model(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            user_labels=batch["user_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"],
            kp_targets=batch["kp_targets"],
            kp_loss_mask=batch["kp_loss_mask"],
        )

        # --------------------------------------------------
        # TRUE USER
        # --------------------------------------------------
        true_user_id = int(out["true_user"].item())
        true_user = idx2user[true_user_id]

        # --------------------------------------------------
        # USER TOP-K
        # --------------------------------------------------
        user_logits = out["user_logits"]          # (1, n_users)
        user_probs = torch.softmax(user_logits, dim=-1)

        top_user_probs, top_user_ids = torch.topk(user_probs, k=TOPK, dim=-1)
        top_user_ids = top_user_ids[0].tolist()

        top_users = [idx2user[uid] for uid in top_user_ids]

        # --------------------------------------------------
        # TRUE LAST NODE
        # --------------------------------------------------
        true_last_node_id = int(batch["node_labels"][0, -1].item())
        true_last_node = idx2node[true_last_node_id]

        # --------------------------------------------------
        # NODE TOP-K
        # --------------------------------------------------
        node_logits = out["last_node_logits"]     # (1, |V|)
        node_probs = torch.softmax(node_logits, dim=-1)

        top_node_probs, top_node_ids = torch.topk(node_probs, k=TOPK, dim=-1)
        top_node_ids = top_node_ids[0].tolist()

        top_nodes = [idx2node[nid] for nid in top_node_ids]

        # --------------------------------------------------
        # KP GOLD
        # --------------------------------------------------
        kp_id = get_kp_supervision_id(row)
        if kp_id is None:
            gold_kps = []
        else:
            _, gold_kps = build_kp_target_from_id(
                kp_id, id2phrases, kp2idx, device
            )

        # --------------------------------------------------
        # KP PRED
        # --------------------------------------------------
        pred_kps_scored = decode_kp_logits(
            out["kp_logits"],
            idx2kp=idx2kp,
            topk=TOPK,
            threshold=0.0
        )[0]

        pred_kps = [kp for kp, _ in pred_kps_scored]

        # --------------------------------------------------
        # RECORD
        # --------------------------------------------------
        records.append({
            "true_userid": true_user,
            "top10_pred_userid": top_users,
            "true_last_node": true_last_node,
            "top10_pred_last_node": top_nodes,
            "true_keyphrases": gold_kps,
            "pred_keyphrases": pred_kps,
        })

results_df = pd.DataFrame(records)
